# USAG-1 (SOSTDC1) Protein Structure Analysis

This notebook visualizes the AlphaFold-predicted structure of USAG-1/SOSTDC1,
the key target for tooth regeneration therapy.

**Background:**
- USAG-1 simultaneously blocks BMP and Wnt signaling, suppressing tooth development
- Anti-USAG-1 antibody TRG-035 is in Phase I clinical trials (Kyoto University, Oct 2024)
- USAG-1 belongs to the sclerostin family with a cystine knot domain
- UniProt: [Q6X4U4](https://www.uniprot.org/uniprotkb/Q6X4U4/entry)
- AlphaFold: [AF-Q6X4U4](https://alphafold.ebi.ac.uk/entry/Q6X4U4)

In [ ]:
# Install dependencies
# !pip install py3Dmol biopython requests matplotlib

In [ ]:
import requests
import os

# Download USAG-1 AlphaFold structure
UNIPROT_ID = "Q6X4U4"
pdb_url = f"https://alphafold.ebi.ac.uk/files/AF-{UNIPROT_ID}-F1-model_v4.pdb"
pdb_path = "../data/usag1_alphafold.pdb"

if not os.path.exists(pdb_path):
    r = requests.get(pdb_url)
    os.makedirs(os.path.dirname(pdb_path), exist_ok=True)
    with open(pdb_path, 'w') as f:
        f.write(r.text)
    print(f"Downloaded USAG-1 structure: {len(r.text)} bytes")
else:
    print(f"Structure already exists at {pdb_path}")

In [ ]:
# Visualize with py3Dmol
import py3Dmol

with open(pdb_path, 'r') as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view.show()

In [ ]:
# Analyze structure: identify cysteine residues (cystine knot)
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure('USAG1', pdb_path)

model = structure[0]
chain = list(model.get_chains())[0]

# Find all cysteine residues
cysteines = []
for residue in chain:
    if residue.get_resname() == 'CYS':
        cysteines.append(residue.get_id()[1])

print(f"Total residues: {len(list(chain.get_residues()))}")
print(f"Cysteine residues ({len(cysteines)}): {cysteines}")
print(f"\nThese cysteines form the cystine knot domain characteristic of the sclerostin family.")

In [ ]:
# Highlight cysteine residues and potential binding regions
view2 = py3Dmol.view(width=800, height=600)
view2.addModel(pdb_data, 'pdb')
view2.setStyle({'cartoon': {'color': 'white', 'opacity': 0.7}})

# Highlight cysteines in yellow
for cys_pos in cysteines:
    view2.addStyle({'resi': cys_pos}, {'stick': {'color': 'yellow'}})

view2.zoomTo()
view2.show()

In [ ]:
# Extract AlphaFold confidence scores (pLDDT) from B-factor column
import matplotlib.pyplot as plt

residue_numbers = []
plddt_scores = []

for residue in chain:
    if residue.get_id()[0] == ' ':  # Standard residues only
        res_num = residue.get_id()[1]
        # pLDDT is stored in B-factor column in AlphaFold PDB files
        ca_atoms = [a for a in residue if a.get_name() == 'CA']
        if ca_atoms:
            plddt = ca_atoms[0].get_bfactor()
            residue_numbers.append(res_num)
            plddt_scores.append(plddt)

plt.figure(figsize=(12, 4))
plt.plot(residue_numbers, plddt_scores, linewidth=1)
plt.axhline(y=90, color='green', linestyle='--', alpha=0.5, label='Very high confidence')
plt.axhline(y=70, color='orange', linestyle='--', alpha=0.5, label='Confident')
plt.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='Low confidence')
plt.xlabel('Residue Number')
plt.ylabel('pLDDT Score')
plt.title('AlphaFold Confidence for USAG-1 (SOSTDC1)')
plt.legend()
plt.tight_layout()
plt.show()

# Identify high-confidence structured regions (potential drug targets)
high_conf = [(r, s) for r, s in zip(residue_numbers, plddt_scores) if s > 80]
print(f"\nHigh-confidence residues (pLDDT > 80): {len(high_conf)} / {len(residue_numbers)}")
print("These well-structured regions are the most suitable targets for computational drug design.")

## Next Steps

1. **Binding site analysis** → See `02_binding_analysis.ipynb`
2. **Molecular docking** → See `03_molecular_docking.ipynb`
3. **Compare with experimental antibody data** from Murashima-Suginami et al. (2021)